# 8j — Preliminary composable forecast (four ways)

Implements `inst/1a_preliminary_framework_plan.md` + the fixes/extensions in `inst/1c`,
`inst/1d`, `inst/1e`: a joint renewal / next-generation-matrix model driven by **age-pair
contact-degree distributions**, scored **four ways** — the 2×2 grid of {unweighted
**NegBin**, weighted **Hurdle-Weibull**} degree models × {**Mean**, **Neighbourhood**} NGM.

The contact **mean** is estimated **per week** with **structural reciprocity**
(`log μ_{i→j} = r + log Nⱼ`) and **spatial-GP smoothing** across the age-pair grid
(separable RBF, shared length-scale; inst/1e) — an independent age-pair GP per window week
(sharing ρ, η; no smoothing over time), so the renewal NGM `N(t)` varies through that week's
`C*ₜ`. Forecasts use the **contact-updated iterate** over **4 origins** × 4 horizons;
**WIS** is computed on a **log scale** and aggregated **by horizon** via R `scoringutils`.

Fitting uses **Pathfinder.jl** (parsimonious fit / init) and optionally **Turing NUTS** (`USE_NUTS`).
See the specs for modelling details and the remaining lean simplifications (no temporal
*correlation* across weeks — RW1/Matérn is the swap-in seam; reduced transmission block).

> **This notebook now does FITTING ONLY.** It fits and caches the MCMC chains (the slow part). Forecast assembly, WIS scoring, and all diagnostic figures moved to `9j_forecast_diagnostics.ipynb`, which reloads these chains.

In [ ]:
ENV["GKSwstype"] = "100"   # headless GR (off-screen PNG) for nbconvert
include("forecast_utils.jl")   # single preamble: base + CoMix pipeline + forecasting framework
using Random, Statistics
mkpath("../res")

USE_NUTS = false   # true ⇒ formal Turing NUTS fit (slow); false ⇒ Pathfinder parsimonious fit

## §1 Window, infection/antibody data, and age-pair degree data

In [ ]:
# `constant_contacts = false` ⇒ contact degree estimated PER WEEK (independent age-pair GP
# each window week, sharing the length-scale ρ and marginal scale η; per-week level cₜ + field
# zₜ; no temporal smoothing). The renewal NGM then varies in time through contacts as well as
# antibody: N(t) uses that week's C*ₜ. (Set true for the pooled one-C*-per-window preliminary.)
cfg  = FrameworkConfig(constant_contacts = false)
grid = cis_age_grid()

# Read the CoMix contact data ONCE and reuse it across every window (avoids re-reading/
# re-joining the full Arrow per origin×horizon). Then roll the forecast origin over the
# whole period the current datasets support ("available period"): each origin needs a
# 12-week fit/lag window back to the first inc2prev week, and contact data out to
# origin+3 for the contact-updated iterate. `available_forecast_origins` derives the range.
raw  = load_raw_contact_inputs()
FORECAST_ORIGINS = available_forecast_origins(cfg; grid = grid, craw = raw.craw)
wins = [WeeklyWindow(o; n_fit = cfg.n_fit, smax = cfg.smax, horizons = cfg.horizons)
        for o in FORECAST_ORIGINS]

println("contact data span  : ", extrema(skipmissing(raw.craw.date)))
println("forecast origins   : ", length(wins), " weekly, ",
        first(FORECAST_ORIGINS), " … ", last(FORECAST_ORIGINS))
let w = wins[1], wd0 = load_window_data(wins[1]; grid = grid)
    println("origin[1] fit weeks: ", w.fit_weeks[1], " … ", w.fit_weeks[end])
    println("weekly infections @ origin[1] (age): ", round.(wd0.I_mean[:, end]; digits = 0))
end

In [ ]:
# Fit config: the four combos and the parallel-fit concurrency (CPU- and memory-balanced).
combos = [(dm, nb) for dm in (NegBinAgePair(), HurdleWeibullAgePair())
                    for nb in (MeanNGM(), NeighbourhoodDegreeNGM())]
MAX_FIT_CONCURRENCY = fit_concurrency()          # min(threads, cores−1, RAM-budget)
if Threads.nthreads() == 1
    @warn "Julia has 1 thread — pre-fit runs sequentially. Start with JULIA_NUM_THREADS>1 " *
          "(e.g. $(max(1, Sys.CPU_THREADS - 1))) for parallel fitting."
end
println("combos = ", length(combos), " | fit concurrency = ", MAX_FIT_CONCURRENCY,
        " | total fits = ", length(wins) * length(combos) * length(cfg.horizons),
        " (cached ones are skipped)")

## §2 Roll over the available period — fit four ways, forecast 1–4 weeks ahead

For **each weekly origin** across the available period the four combos are fit and forecast.
The contact **mean** is a **reciprocity-structural, GP-smoothed** field (one symmetric
log-rate per unordered age pair, separable-RBF smoothing over age midpoints 70+→74.5,
`log μ_{i→j}=r+log Nⱼ` ⟹ exact reciprocity). Forecasting is the **contact-updated iterate**
(inst/1d): per origin t₀ and horizon `h`, the degree window ends at `t₀+h−1` (infections/
antibody frozen at t₀), the NGM is refreshed and one renewal step taken.

The loop is **memory-bounded**: per origin it builds only that origin's 4 degree windows
(reusing the single raw read), **parallel-pre-fits** its 16 chains (`prefit_chains!`, bounded
to `MAX_FIT_CONCURRENCY` simultaneous fits), assembles the forecasts, then discards the degree
data. Chains are cached per (origin, horizon) under `../dt_intermediate/8j_chn_*.jld2`, so the
run is **resumable** — a re-run reloads finished chains and only fits what's missing.

In [ ]:
# Parallel, resumable PRE-FIT ONLY. For each weekly origin build the window data and the 4
# contact/degree windows, then fit this origin's 16 chains (4 combos × 4 horizons) concurrently
# and cache each to ../dt_intermediate/8j_chn_<degree>_<ngm>_<contacts>_<origin>_h<h>.jld2.
# Cached chains are skipped, so re-runs only fit what's missing. Forecast assembly, scoring, and
# diagnostics live in 9j_forecast_diagnostics.ipynb (it reloads these chains — run this first).
t0 = time()
for (oi, win_o) in enumerate(wins)
    wd_o  = load_window_data(win_o; grid = grid)
    # this origin's 4 contact/degree windows (reuse the single raw read); discarded after.
    apd_o = [prepare_degree_data(
                 WeeklyWindow(win_o.origin + Day(7 * (h - 1));
                              n_fit = cfg.n_fit, smax = cfg.smax, horizons = cfg.horizons),
                 cfg; grid = grid, setting = :all,
                 df_part_raw = raw.df_part, craw_raw = raw.craw)
             for h in cfg.horizons]
    # parallel pre-fit this origin's 16 chains (cached ones skipped)
    prefit_chains!(combos, [win_o], [wd_o], cfg, [apd_o];
                   grid = grid, setting = :all, use_nuts = USE_NUTS,
                   save_dir = "../dt_intermediate", max_concurrent = MAX_FIT_CONCURRENCY)
    if oi % 5 == 0 || oi == length(wins)
        println("  origin $oi/$(length(wins)) (", win_o.origin, ")  elapsed ",
                round(Int, time() - t0), "s")
    end
end

# Verifiable tail: count how many of this run's chain-cache files are present.
chain_paths = ["../dt_intermediate/8j_chn_$(degree_label(dm))_$(ngm_label(nb))_" *
               "$(contacts_label(cfg))_$(win.origin)_h$(h).jld2"
               for win in wins, (dm, nb) in combos, h in cfg.horizons]
n_have, n_expect = count(isfile, chain_paths), length(chain_paths)
println("cached chains: $n_have / $n_expect present under ../dt_intermediate ",
        "(", contacts_label(cfg), " contacts) — feed 9j_forecast_diagnostics.ipynb")